In [31]:
from neuralbench.registry import ALL_MODELS, TASKS
print("EEG tasks:", ", ".join(TASKS["eeg"]))
print("Available models:", ", ".join(ALL_MODELS))

classic_models = [
    "shallow_fbcsp_net",
    "simpleconv_time_agg",
    "eegnet",
    "deep4net",
    "eegconformer",
    "atcnet",
    "bdtcn",
    "ctnet",
]

foundation_models = [
    "bendr",
    "biot",
    "cbramod",
    "labram",
    "luna",
    "reve",
]

print("Classic models:")
for model in classic_models:
    print("  ", model)

print("\nFoundation models:")
for model in foundation_models:
    print("  ", model)

EEG tasks: age, artifact, audiovisual_stimulus, clinical_event, cvep, dementia_diagnosis, depression_diagnosis, emotion, ern, image, lrp, mental_arithmetic, mental_imagery, mental_workload, mismatch_negativity, motor_execution, motor_imagery, n170, n2pc, n400, p3, parkinsons_diagnosis, pathology, psychopathology, reaction_time, schizophrenia_diagnosis, seizure, sentence, sex, sleep_arousal, sleep_onset, sleep_stage, speech, ssvep, typing, video, word
Available models: atcnet, bdtcn, bendr, biot, cbramod, chance, cospectra_log_lr, cov_ts_lr, cov_ts_ridge, ctnet, deep4net, dummy, eegconformer, eegnet, emg2qwerty, fmri_linear, fmri_mlp, labram, luna, mae, neuropose, reve, sensingdynamics, shallow_fbcsp_net, simpleconv, simpleconv_time_agg, vemg2pose, xdawn_ts_lr
Classic models:
   shallow_fbcsp_net
   simpleconv_time_agg
   eegnet
   deep4net
   eegconformer
   atcnet
   bdtcn
   ctnet

Foundation models:
   bendr
   biot
   cbramod
   labram
   luna
   reve


In [ ]:
from neuralbench import run_benchmark
results = run_benchmark(device = 'eeg',
                        task = 'audiovisual_stimulus', 
                        model = ['deep4net', 'eegnet'])

In [19]:
results = run_benchmark(
    device="eeg",
    task="audiovisual_stimulus",
    model = ["eegnet", "deep4net"],
    plot_cached=True,
)

INFO:neuralbench.cli:--- PREPARING GLOBAL PLOTS AND TABLES ---
INFO:neuralbench.aggregator:Saved computational stats to D:\Foundation Challenge 2026\results\outputs\other\computational_stats.json
Generating plots:  30%|███       | 3/10 [00:02<00:04,  1.74it/s]c:\Users\mcapo\anaconda3\envs\foundation2026\Lib\site-packages\neuralbench\plots\normalized_summary.py:305: UserWarning: Skipping normalized-lines-summary plot: no tasks with baseline data
  warnings.warn(
Generating plots:  60%|██████    | 6/10 [00:02<00:00,  4.60it/s]c:\Users\mcapo\anaconda3\envs\foundation2026\Lib\site-packages\neuralbench\plots\normalized_summary.py:305: UserWarning: Skipping normalized-lines-summary plot: no tasks with baseline data
  warnings.warn(
Generating plots: 100%|██████████| 10/10 [00:02<00:00,  4.63it/s]
INFO:neuralbench.plots.adaptation:plot_adaptation_comparison: no foundation-model rows -- skipping.



Outputs saved under D:\Foundation Challenge 2026\results\outputs in subfolders ('core', 'full', 'other', 'adaptation')


In [ ]:
# 2. Build the caches under CACHE_DIR (~13 GB): the preprocessed windows,
#    plus one frozen DINOv2-giant embedding per unique stimulus (~100 MB
#    for THINGS-EEG2, content-keyed and shared with the other image
#    tasks). The only --prepare of the four tracks that needs a GPU.
#    ~15 min for eegnet's cache, ~45 min for reve's, and ~10 min to embed
#    the 16740 stimuli, spread over 10 and 128 SLURM jobs respectively.
neuralbench eeg image --prepare

# 3. Sanity check before you queue anything: 2 epochs, a data subset, one
#    seed, always in-process, so progress lands in your terminal. ~2 min
#    on one V100 with the cache warm. Name the model you actually plan to
#    run -- a bare --debug takes the config default, which is EEGNet.
neuralbench eeg image -m eegnet --debug

# 4. Same check for the foundation model. REVE's weights are gated on the
#    HuggingFace Hub, so this needs an account and an accepted licence;
#    it is the cheapest place to discover that, because a queued run
#    reports the failure into a job log instead of your terminal.
neuralbench eeg image -m reve --debug

# 5. Full baseline -- task-specific model (EEGNet). ~2.5 h per seed, and
#    the default grid is three seeds (concurrent on SLURM).
neuralbench eeg image -m eegnet

# 6. Full baseline -- foundation model (REVE), fine-tuned end to end.
#    ~5.5 h per seed. ~69M parameters against EEGNet's ~1.5k, all of them
#    trainable here, so this one wants a datacentre GPU rather than a
#    laptop; it also preprocesses at 200 Hz against the 120 Hz default,
#    warming a second cache.
neuralbench eeg image -m reve